# Estadística Descriptiva y Análisis Univariado
## Notebook de Clase (Demostración)

Este notebook acompaña la presentación teórica. Contiene código de demostración para ilustrar los conceptos clave **sin** resolver los ejercicios de los alumnos.

**Contenidos:**
1. Distribución univariada y medidas de tendencia central / dispersión
2. Visualización univariada
3. Variables categóricas
4. Detección de outliers (IQR y Z-score)
5. Transformaciones univariadas
6. Pruebas estadísticas básicas (introducción)
7. Correlación

---


In [ ]:
# Librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(42)

# Datos sintéticos para demostración
n = 500
# Variable aproximadamente normal
x_normal = np.random.normal(loc=50, scale=10, size=n)
# Variable con sesgo positivo (ingresos simulados)
x_skew = np.random.lognormal(mean=3.5, sigma=0.8, size=n)
# Variable categórica
cats = np.random.choice(["A", "B", "C", "D"], size=n, p=[0.4, 0.3, 0.2, 0.1])

df = pd.DataFrame({
    "normal": x_normal,
    "sesgada": x_skew,
    "categoria": cats
})
df.head()


## 1. Medidas de tendencia central y dispersión


In [ ]:
def resumen_numerico(serie, nombre="Variable"):
    '''Imprime un resumen completo de una serie numérica.'''
    print(f"=== {nombre} ===")
    print(f"Media          : {serie.mean():.3f}")
    print(f"Mediana        : {serie.median():.3f}")
    print(f"Moda           : {serie.mode().iloc[0]:.3f}" if not serie.mode().empty else "Moda: —")
    print(f"Desv. estándar : {serie.std():.3f}")
    print(f"Varianza       : {serie.var():.3f}")
    print(f"IQR            : {serie.quantile(0.75) - serie.quantile(0.25):.3f}")
    print(f"Mín / Máx      : {serie.min():.3f} / {serie.max():.3f}")
    print(f"Sesgo (skew)   : {serie.skew():.3f}")
    print(f"Curtosis       : {serie.kurtosis():.3f}")
    print()

resumen_numerico(df["normal"], "Variable aproximadamente normal")
resumen_numerico(df["sesgada"], "Variable con sesgo positivo")


## 2. Visualización univariada


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Histograma + KDE
sns.histplot(df["normal"], kde=True, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Histograma + KDE (normal)")

sns.histplot(df["sesgada"], kde=True, ax=axes[0, 1], color="coral")
axes[0, 1].set_title("Histograma + KDE (sesgada)")

# Boxplots
sns.boxplot(y=df["normal"], ax=axes[1, 0], color="steelblue")
axes[1, 0].set_title("Boxplot (normal)")

sns.boxplot(y=df["sesgada"], ax=axes[1, 1], color="coral")
axes[1, 1].set_title("Boxplot (sesgada)")

plt.tight_layout()
plt.show()


## 3. Variables categóricas


In [ ]:
# Tabla de frecuencias y proporciones
freq = df["categoria"].value_counts().sort_index()
prop = df["categoria"].value_counts(normalize=True).sort_index()
tabla = pd.DataFrame({"Frecuencia": freq, "Proporción": prop.round(3)})
print(tabla)

# Countplot
plt.figure(figsize=(7, 4))
sns.countplot(x="categoria", data=df, order=["A", "B", "C", "D"], palette="viridis")
plt.title("Frecuencia de categorías")
plt.show()


## 4. Detección de outliers

### Método IQR


In [ ]:
def detectar_outliers_iqr(serie, factor=1.5):
    '''Devuelve máscara booleana de outliers según el método IQR.'''
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - factor * iqr
    limite_sup = q3 + factor * iqr
    return (serie < limite_inf) | (serie > limite_sup)

out_iqr = detectar_outliers_iqr(df["sesgada"])
print(f"Outliers detectados (IQR): {out_iqr.sum()} ({100*out_iqr.mean():.1f}%)")
print(f"Límites: [{df['sesgada'].quantile(0.25) - 1.5*(df['sesgada'].quantile(0.75)-df['sesgada'].quantile(0.25)):.2f}, "
      f"{df['sesgada'].quantile(0.75) + 1.5*(df['sesgada'].quantile(0.75)-df['sesgada'].quantile(0.25)):.2f}]")


### Método Z-score


In [ ]:
def detectar_outliers_zscore(serie, umbral=3.0):
    '''Devuelve máscara booleana de outliers según |Z| > umbral.'''
    z = (serie - serie.mean()) / serie.std()
    return np.abs(z) > umbral

out_z = detectar_outliers_zscore(df["sesgada"])
print(f"Outliers detectados (Z-score > 3): {out_z.sum()} ({100*out_z.mean():.1f}%)")


## 5. Transformaciones univariadas


In [ ]:
# Transformación logarítmica
df["sesgada_log"] = np.log(df["sesgada"])

# Box-Cox (requiere valores positivos)
from scipy.stats import boxcox
df["sesgada_bc"], lambda_bc = boxcox(df["sesgada"])
print(f"Lambda óptimo de Box-Cox: {lambda_bc:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df["sesgada"], kde=True, ax=axes[0], color="coral")
axes[0].set_title("Original (sesgada)")
sns.histplot(df["sesgada_log"], kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("Log")
sns.histplot(df["sesgada_bc"], kde=True, ax=axes[2], color="seagreen")
axes[2].set_title(f"Box-Cox (λ={lambda_bc:.2f})")
plt.tight_layout()
plt.show()

print("Sesgo original :", df["sesgada"].skew().round(3))
print("Sesgo log      :", df["sesgada_log"].skew().round(3))
print("Sesgo Box-Cox  :", df["sesgada_bc"].skew().round(3))


## 6. Introducción a pruebas estadísticas

Demostración rápida de un t-test y de Mann-Whitney (sin profundizar en interpretación de negocio).


In [ ]:
# Dos grupos sintéticos
grupo_a = np.random.normal(50, 10, 80)
grupo_b = np.random.normal(55, 12, 80)

# t-test de Welch (no asume igualdad de varianzas)
t_stat, p_val = stats.ttest_ind(grupo_a, grupo_b, equal_var=False)
print(f"t-test de Welch: t = {t_stat:.3f}, p = {p_val:.4f}")

# Mann-Whitney
u_stat, p_mw = stats.mannwhitneyu(grupo_a, grupo_b, alternative="two-sided")
print(f"Mann-Whitney U: U = {u_stat:.1f}, p = {p_mw:.4f}")


## 7. Correlación


In [ ]:
# Creamos una segunda variable correlacionada
df["relacionada"] = 0.7 * df["normal"] + np.random.normal(0, 5, n)

print("Pearson :", df["normal"].corr(df["relacionada"], method="pearson").round(3))
print("Spearman:", df["normal"].corr(df["relacionada"], method="spearman").round(3))
print("Kendall :", df["normal"].corr(df["relacionada"], method="kendall").round(3))

sns.jointplot(data=df, x="normal", y="relacionada", kind="reg", height=5)
plt.show()


---
**Fin del notebook de clase.**  
Los ejercicios para los alumnos se encuentran en el notebook `Ejercicios_Estadistica_Univariada.ipynb`.
